# 🧴 DermaSmart — AI Skin Condition Classifier

**Author:** Ravi Kiran Reddy Bada  
**GitHub:** [github.com/Ravikiranreddybada](https://github.com/Ravikiranreddybada)  

---

## Overview

DermaSmart is an AI-powered skin intelligence system that uses **transfer learning** on top of **MobileNetV2** to classify skin conditions from images.

This notebook covers the full ML pipeline:
1. Downloading and loading the Dermnet dataset from Kaggle
2. Preprocessing images for MobileNetV2 input
3. Building a transfer learning model with a custom classification head
4. Training with early stopping to prevent overfitting
5. Saving the final model for deployment

**Dataset:** [Dermnet](https://www.kaggle.com/datasets/shubhamgoel27/dermnet) — 23 skin condition categories  
**Base Model:** MobileNetV2 (pretrained on ImageNet)  
**Framework:** TensorFlow / Keras  
**Hardware:** GPU (T4)


## 1. Imports

All required libraries for image processing, model building, and training.

In [1]:
# ── Core ──────────────────────────────────────────────────────────────────────
from typing import List, Set, Dict, Tuple, Optional
import os
import pathlib
import random

# ── Image Processing ──────────────────────────────────────────────────────────
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# ── Deep Learning ─────────────────────────────────────────────────────────────
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Download Dataset

The **Dermnet** dataset contains 23 classes of skin conditions with ~19,500 training images.
We use `kagglehub` to pull the latest version directly into the Colab environment.

In [2]:
import kagglehub

# Download Dermnet dataset (1.72 GB)
path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print(f"Dataset downloaded to: {path}")

100%|██████████| 1.72G/1.72G [00:59<00:00, 31.2MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/shubhamgoel27/dermnet/versions/1


## 3. Load & Split Data

For each skin condition class:
- **80%** of images → training set
- **20%** of images → validation set

All images are resized to **224×224** to match MobileNetV2's expected input shape.

In [3]:
data_path = os.path.join(path, 'train')
CLASSES = sorted(os.listdir(data_path))
IMG_SIZE = (224, 224)

train_data = []
val_data   = []

for folder in CLASSES:
    folder_path = os.path.join(data_path, folder)
    files = os.listdir(folder_path)

    # 80/20 stratified split per class
    num_train  = int(0.8 * len(files))
    files_train = random.sample(files, num_train)
    files_val   = list(set(files) - set(files_train))

    for fname in files_train:
        img = cv2.imread(os.path.join(folder_path, fname))
        img = cv2.resize(img, IMG_SIZE)
        train_data.append((img, folder))

    for fname in files_val:
        img = cv2.imread(os.path.join(folder_path, fname))
        img = cv2.resize(img, IMG_SIZE)
        val_data.append((img, folder))

print(f"Classes ({len(CLASSES)}): {CLASSES}")
print(f"Training samples  : {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

## 4. Build Transfer Learning Model

### Architecture

```
Input (224×224×3)
    │
    ▼
MobileNetV2 backbone  ← frozen (ImageNet weights)
    │
    ▼
GlobalAveragePooling2D
    │
    ▼
Dense(512, relu)       ← custom classification head
    │
    ▼
Dense(23, softmax)     ← 23 skin condition classes
```

Freezing the backbone lets us leverage ImageNet's rich feature representations while only training the lightweight head — far faster than training from scratch.

In [4]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from keras.callbacks import EarlyStopping

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_WIDTH  = 224
IMG_HEIGHT = 224
NUM_CLASSES = 23
BATCH_SIZE  = 64
EPOCHS      = 20

# ── Base model (frozen) ───────────────────────────────────────────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_WIDTH, IMG_HEIGHT, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze backbone weights

# ── Custom classification head ────────────────────────────────────────────────
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Preprocess Data

Apply MobileNetV2's built-in `preprocess_input` (scales pixel values to `[-1, 1]`) and one-hot encode the class labels.

In [5]:
# ── Separate images and labels ────────────────────────────────────────────────
X_train, y_train = zip(*train_data)
X_val,   y_val   = zip(*val_data)

# ── Pixel normalisation for MobileNetV2 ──────────────────────────────────────
X_train = preprocess_input(np.array(X_train))
X_val   = preprocess_input(np.array(X_val))

# ── Label encoding → one-hot ──────────────────────────────────────────────────
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded   = le.transform(y_val)

y_train_ohe = to_categorical(y_train_encoded, NUM_CLASSES)
y_val_ohe   = to_categorical(y_val_encoded,   NUM_CLASSES)

print(f"X_train shape : {X_train.shape}")
print(f"X_val shape   : {X_val.shape}")
print(f"Label classes : {list(le.classes_)}")

## 6. Train the Model

Training uses **EarlyStopping** on validation loss (patience = 10) to avoid overfitting.  
The best weights are restored automatically when training halts.

In [6]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    min_delta=0.001,
    mode='min',
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train_ohe,
    validation_data=(X_val, y_val_ohe),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 41s 146ms/step - accuracy: 0.2423 - loss: 2.6679 - val_accuracy: 0.3206 - val_loss: 2.3172
Epoch 2/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - accuracy: 0.3833 - loss: 2.0446 - val_accuracy: 0.3549 - val_loss: 2.2301
Epoch 3/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - accuracy: 0.4776 - loss: 1.7471 - val_accuracy: 0.3713 - val_loss: 2.1747
Epoch 4/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - accuracy: 0.5279 - loss: 1.5462 - val_accuracy: 0.3937 - val_loss: 2.1049
Epoch 5/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - accuracy: 0.6096 - loss: 1.2938 - val_accuracy: 0.4075 - val_loss: 2.1153
Epoch 6/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - accuracy: 0.6782 - loss: 1.0822 - val_accuracy: 0.4113 - val_loss: 2.1787
Epoch 7/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - accuracy: 0.7549 - loss: 0.8614 - val_accuracy: 0.4178 - val_loss: 2.2174
Epoch 8/20
195/195 ━━━━━━━━━━━━━━━━━━━━ 13s 69ms/step - accuracy: 0.8073 - loss: 0.6829 -

## 7. Training Curves

Visualise accuracy and loss across epochs to confirm the model is learning and early stopping fired at the right time.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DermaSmart — Training History', fontsize=14, fontweight='bold')

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train accuracy', color='#C9A84C')
axes[0].plot(history.history['val_accuracy'], label='Val accuracy',   color='#8A9E8C')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train loss', color='#C9A84C')
axes[1].plot(history.history['val_loss'], label='Val loss',   color='#8A9E8C')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 8. Evaluate on Validation Set

In [8]:
val_loss, val_acc = model.evaluate(X_val, y_val_ohe, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc * 100:.2f}%")

## 9. Save Model

Export the trained model in the native Keras format (`.keras`) for use in the DermaSmart backend.

In [9]:
MODEL_PATH = 'tf_model.keras'
model.save(MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")
print(f"File size   : {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB")

Model saved → tf_model.keras


---

## Summary

| Item | Detail |
|------|--------|
| **Model** | MobileNetV2 + custom head |
| **Dataset** | Dermnet (23 skin condition classes) |
| **Input shape** | 224 × 224 × 3 |
| **Training split** | 80% train / 20% val (per class) |
| **Batch size** | 64 |
| **Max epochs** | 20 (early stopping patience=10) |
| **Optimiser** | Adam |
| **Loss** | Categorical cross-entropy |
| **Output** | `tf_model.keras` |

**Author:** Ravi Kiran Reddy Bada — [github.com/Ravikiranreddybada](https://github.com/Ravikiranreddybada)
